In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

def load_urls(file_path):
    urls = []
    if not os.path.exists(file_path):
        return urls

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if line.startswith("http"):
                urls.append(line)

    return urls


def scrape_global_firepower_data():

    base_page = "https://www.globalfirepower.com/countries-listing.php"
    metric_pages = load_urls("links_for_military_data.txt")

    response = requests.get(base_page, headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")

    country_blocks = soup.select("div.picTrans.recordsetContainer")

    countries = []
    ranks = []

    for block in country_blocks:
        try:
            country = block.find("span", class_="textWhite textLarge textShadow").text.strip()
            rank = block.find("span", class_="textWhite textLarge textBold").text.strip()
            countries.append(country)
            ranks.append(rank)
        except:
            continue

    data_df = pd.DataFrame({
        "Country": countries,
        "Rank": ranks
    })

    for page in metric_pages:
        try:
            response = requests.get(page, headers=HEADERS)
            soup = BeautifulSoup(response.text, "html.parser")

            metric_blocks = soup.select("div.picTrans.recordsetContainer")
            if not metric_blocks:
                continue

            temp_country = []
            temp_value = []

            for block in metric_blocks:
                try:
                    country = block.find("span", class_="textWhite textLarge textShadow").text.strip()
                    value = block.find_all("span", class_="textWhite textLarge")[-1].text.strip()
                    temp_country.append(country)
                    temp_value.append(value)
                except:
                    continue

            metric_name = page.split("/")[-1].replace(".php", "").replace("-", "_")

            metric_df = pd.DataFrame({
                "Country": temp_country,
                metric_name: temp_value
            })

            data_df = data_df.merge(metric_df, on="Country", how="left")

        except:
            continue

    for column in data_df.columns[2:]:
        data_df[column] = (
            data_df[column]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.extract(r"(\d+(?:\.\d+)?)")[0]
        )

    return data_df


if __name__ == "__main__":
    final_df = scrape_global_firepower_data()
    final_df.to_csv("military_raw_data.csv", index=False)
    print("global_military_data_final.csv created successfully!")


global_military_data_final.csv created successfully!
